# Do the five CFPB dimensions carry distinct information?

## Question

FinePrint shows five dimension scores per lender. If those five scores are the same signal
measured five times, presenting all five implies five independent pieces of evidence and is
misleading. If they are genuinely distinct, showing them separately is honest and useful.

This notebook measures the dependence between the five dimensions and decides which case we
are in. It uses the **Method C posterior peer-probability scores** exactly as validated in
`notebooks/cfpb_dimension_scoring.ipynb`, and it does not redesign them.

## The central complication, stated up front

The five complaint shares are cut from the same complaint total, so they cannot all be large
at once. Part of any negative correlation between dimension scores is therefore **arithmetic,
not behavioural**: a lender that looks bad on fees mechanically has less complaint mass left
for servicing. This notebook separates that from real overlap before drawing any conclusion,
because the two point in opposite directions - closure *hides* redundancy, and averaging
across dimensions is hurt by the negative part.

## Constraints observed

No overall score, no weights, no lender ranking or grade, no composite of any kind, and no
per-lender aggregate across the five dimensions is computed or displayed. Lender-level rows
appear only as side-by-side dimension-by-dimension inspection. No file is written; the
processed CSV is read-only here.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import beta as beta_dist, pearsonr, spearmanr

ROOT = Path("/Users/mihirkale/cdc")
IN = ROOT / "data/processed/payday_shrunk_features.csv"

# Dimension slugs, exactly as used in cfpb_dimension_scoring.ipynb
SLUG = ["withdrawal", "fees", "unauthorized", "credit_rep", "servicing"]
DIMS = {
    "withdrawal": "withdrawal_and_payment_control",
    "fees": "fees_and_costs",
    "unauthorized": "unauthorized_or_unrequested_loan",
    "credit_rep": "credit_reporting",
    "servicing": "servicing_and_payment_handling",
}
SHORT = {s: (DIMS[s][:4] if s != "credit_rep" else "cred") for s in SLUG}

pd.set_option("display.width", 240)

t = pd.read_csv(IN)
n = t["n_complaints"].to_numpy(float)
L = len(t)

print(f"input             {IN.name}")
print(f"lenders           {L}")
print(f"total complaints  {int(n.sum()):,}")
_in5 = t[[f"n_{s}" for s in SLUG]].to_numpy(float).sum()
_shr = t[[f"share_{s}" for s in SLUG]].to_numpy(float).sum(axis=1)
print(f"\nthe five dimensions cover {100*_in5/n.sum():.1f}% of complaints, so the five")
print(f"shares sum to at most 1 and average {_shr.mean():.3f}. That closure is the source")
print("of the mechanical negative correlation diagnosed in section 3.")
print("processed CSV is read-only in this notebook; nothing is written back.")

input             payday_shrunk_features.csv
lenders           482
total complaints  6,024

the five dimensions cover 82.3% of complaints, so the five
shares sum to at most 1 and average 0.781. That closure is the source
of the mechanical negative correlation diagnosed in section 3.
processed CSV is read-only in this notebook; nothing is written back.


## 1. Reproduce Method C, unchanged

Identical to the validated definition: a Beta posterior per lender and dimension, referenced
to the **median of the fitted Beta** for that dimension.

$$\theta_{ic} \mid x_{ic}, n_i \sim \mathrm{Beta}(x_{ic}+\alpha_c,\ n_i - x_{ic} + \beta_c)$$
$$\mathrm{Score}_{ic} = 100 \cdot P(\theta_{ic} < \theta^{\text{peer}}_c), \qquad
  \theta^{\text{peer}}_c = \mathrm{median}(\mathrm{Beta}(\alpha_c,\beta_c))$$

The reference is a property of the fitted model, so these scores do not depend on which
subset of lenders is used as the peer group. Section 7 asserts the reproduction against
specific values already published in the scoring notebook.

In [2]:
def prior_ab(s):
    return float(t[f"prior_alpha_{s}"].iloc[0]), float(t[f"prior_beta_{s}"].iloc[0])

def peer_rate(s):
    a, b = prior_ab(s)
    return float(beta_dist.median(a, b))

def score_C(s):
    a, b = prior_ab(s)
    x = t[f"n_{s}"].to_numpy(float)
    return 100.0 * beta_dist.cdf(peer_rate(s), x + a, n - x + b)

def latent_z(s):
    # The signed posterior z-score of the peer rate. score_C is a monotone,
    # NONLINEAR squashing of this quantity, which matters in section 3.
    a, b = prior_ab(s)
    x = t[f"n_{s}"].to_numpy(float)
    A_, B_ = x + a, n - x + b
    return (peer_rate(s) - A_ / (A_ + B_)) / np.sqrt(A_ * B_ / ((A_ + B_) ** 2 * (A_ + B_ + 1)))

S = pd.DataFrame({s: score_C(s) for s in SLUG}, index=t.index)
Z = pd.DataFrame({s: latent_z(s) for s in SLUG}, index=t.index)
P = pd.DataFrame({s: t[f"post_{s}"] for s in SLUG}, index=t.index)
W = pd.DataFrame({s: t[f"share_{s}"] for s in SLUG}, index=t.index)
X = pd.DataFrame({s: t[f"n_{s}"] for s in SLUG}, index=t.index)

print("peer reference rates (fitted Beta median) and score summaries")
for s in SLUG:
    a, b = prior_ab(s)
    print(f"  {DIMS[s]:<34} peer={peer_rate(s):.4f}  alpha={a:5.3f} beta={b:6.3f}  "
          f"C mean={S[s].mean():5.2f} median={S[s].median():5.2f} "
          f"IQR={S[s].quantile(.75)-S[s].quantile(.25):5.2f}")
print(f"\nall five score columns are finite and bounded: "
      f"{bool(np.isfinite(S.to_numpy()).all() and (S.to_numpy() >= 0).all() and (S.to_numpy() <= 100).all())}")

peer reference rates (fitted Beta median) and score summaries
  withdrawal_and_payment_control     peer=0.0684  alpha=0.556 beta= 4.114  C mean=50.35 median=55.45 IQR=19.88
  fees_and_costs                     peer=0.2075  alpha=1.332 beta= 4.218  C mean=50.83 median=58.75 IQR=36.29
  unauthorized_or_unrequested_loan   peer=0.0954  alpha=1.489 beta=11.451  C mean=50.26 median=53.68 IQR=17.22
  credit_reporting                   peer=0.0406  alpha=0.610 beta= 8.016  C mean=50.26 median=53.03 IQR= 5.35
  servicing_and_payment_handling     peer=0.2414  alpha=2.868 beta= 8.324  C mean=50.17 median=56.74 IQR=31.91

all five score columns are finite and bounded: True


## 2. Pairwise dependence across four volume subsets

Pearson and Spearman for all 10 pairs, on four nested subsets: all 482 lenders, and the
lenders with at least 10, 20 and 50 complaints.

The subsets matter. Method C deliberately compresses uncertain lenders toward 50, and 382 of
the 482 lenders have fewer than 10 complaints, so the full-sample correlation is computed
mostly from lenders whose scores are all near 50 and nearly independent **by construction**.
That is a property of the score, not evidence that the dimensions are unrelated. The
high-volume subsets, where the scores actually discriminate, are the informative ones.

The 50+ subset has only 32 lenders, so its correlations carry wide uncertainty; Fisher-z
confidence intervals are reported alongside and one pair is shown to be unstable.

In [3]:
SUBSETS = [("all 482", np.ones(L, bool)),
            (">=10 complaints", n >= 10),
            (">=20 complaints", n >= 20),
            (">=50 complaints", n >= 50)]

def pair_index():
    iu = np.triu_indices(len(SLUG), 1)
    return iu, [f"{SHORT[SLUG[a]]}/{SHORT[SLUG[b]]}" for a, b in zip(*iu)]

IU, PAIR = pair_index()

def fisher_ci(r, k):
    # 95% CI for a Pearson correlation via the Fisher z transform.
    if k < 4 or abs(r) >= 1.0:
        return (np.nan, np.nan)
    z = np.arctanh(r)
    se = 1.0 / np.sqrt(k - 3)
    return tuple(np.tanh(z + np.array([-1, 1]) * 1.96 * se))

for lab, m in SUBSETS:
    k = int(m.sum())
    pear = S[m].corr().to_numpy()
    spear = S[m].corr(method="spearman").to_numpy()
    print(f"\n{'='*74}\n{lab}   (k = {k})\n{'='*74}")
    print("Pearson")
    print(pd.DataFrame(pear, index=[SHORT[s] for s in SLUG],
                       columns=[SHORT[s] for s in SLUG]).round(3).to_string())
    print("Spearman")
    print(pd.DataFrame(spear, index=[SHORT[s] for s in SLUG],
                       columns=[SHORT[s] for s in SLUG]).round(3).to_string())
    print("Pearson 95% CI (Fisher z), off-diagonal pairs")
    rows = []
    for (a, b), pr in zip(zip(*IU), PAIR):
        lo, hi = fisher_ci(pear[a, b], k)
        rows.append({"pair": pr, "r": pear[a, b],
                     "lo": lo, "hi": hi,
                     "sp": spear[a, b],
                     "ci_width": hi - lo,
                     "sign_stable": None})
    d = pd.DataFrame(rows)
    print(d.round(3).to_string(index=False))


all 482   (k = 482)
Pearson
       with   fees   unau   cred   serv
with  1.000 -0.124 -0.222 -0.287 -0.003
fees -0.124  1.000 -0.339 -0.295 -0.481
unau -0.222 -0.339  1.000  0.305  0.142
cred -0.287 -0.295  0.305  1.000  0.130
serv -0.003 -0.481  0.142  0.130  1.000
Spearman
       with   fees   unau   cred   serv
with  1.000 -0.091 -0.146 -0.139  0.006
fees -0.091  1.000 -0.245 -0.209 -0.397
unau -0.146 -0.245  1.000  0.351  0.060
cred -0.139 -0.209  0.351  1.000  0.077
serv  0.006 -0.397  0.060  0.077  1.000
Pearson 95% CI (Fisher z), off-diagonal pairs
     pair      r     lo     hi     sp  ci_width sign_stable
with/fees -0.124 -0.211 -0.035 -0.091     0.176        None
with/unau -0.222 -0.306 -0.136 -0.146     0.170        None
with/cred -0.287 -0.367 -0.202 -0.139     0.164        None
with/serv -0.003 -0.092  0.086  0.006     0.179        None
fees/unau -0.339 -0.416 -0.258 -0.245     0.158        None
fees/cred -0.295 -0.374 -0.211 -0.209     0.163        None
fees/serv -0.481

### 2a. Sign stability across subsets

A pair is only a redundancy candidate if it is large **and** keeps its sign and magnitude as
the sample changes. Two pairs qualify on magnitude: `unau/cred` (positive, and it *grows* with
volume) and `fees/serv` (negative, and it also grows). One pair, `with/unau`, is unstable: its
Pearson correlation decays from -0.26 to -0.04 as the subset tightens, which is what a noisy
32-lender correlation looks like. It is not flagged.

In [4]:
stab = []
for (a, b), pr in zip(zip(*IU), PAIR):
    rs = [S[m].corr().iloc[a, b] for _, m in SUBSETS]
    sp = [S[m].corr(method="spearman").iloc[a, b] for _, m in SUBSETS]
    signs = {np.sign(v) for v in rs if abs(v) > 0.05}
    stab.append({"pair": pr,
                 "r_all": rs[0], "r_ge10": rs[1], "r_ge20": rs[2], "r_ge50": rs[3],
                 "sp_all": sp[0], "sp_ge50": sp[3],
                 "min_abs_r": min(abs(v) for v in rs),
                 "sign_stable": len(signs) == 1})
stab = pd.DataFrame(stab).sort_values("min_abs_r", ascending=False)
print(stab.round(3).to_string(index=False))
print("\nPairs with |r| >= 0.30 in EVERY subset and a stable sign:")
strong = stab[(stab.min_abs_r >= 0.30) & stab.sign_stable]
for _, r in strong.iterrows():
    print(f"  {r['pair']:<14} r from {r['r_all']:+.2f} to {r['r_ge50']:+.2f}")
print("\nThe C-score correlation is a *bounded* transform, so it also compresses the extreme")
print("lenders toward 50 and understates the latent dependence. Section 3b quantifies that,")
print("and it is the reason the substantive overlap is larger than section 2 suggests.")

     pair  r_all  r_ge10  r_ge20  r_ge50  sp_all  sp_ge50  min_abs_r  sign_stable
fees/serv -0.481  -0.619  -0.641  -0.698  -0.397   -0.695      0.481         True
fees/unau -0.339  -0.417  -0.459  -0.408  -0.245   -0.372      0.339         True
unau/cred  0.305   0.505   0.535   0.555   0.351    0.663      0.305         True
with/cred -0.287  -0.347  -0.342  -0.321  -0.139   -0.359      0.287         True
fees/cred -0.295  -0.333  -0.330  -0.212  -0.209   -0.399      0.212         True
unau/serv  0.142   0.381   0.442   0.448   0.060    0.457      0.142         True
cred/serv  0.130   0.298   0.324   0.402   0.077    0.614      0.130         True
with/fees -0.124  -0.151  -0.209  -0.370  -0.091   -0.441      0.124         True
with/unau -0.222  -0.274  -0.256  -0.043  -0.146   -0.125      0.043         True
with/serv -0.003   0.120   0.165   0.166   0.006    0.110      0.003         True

Pairs with |r| >= 0.30 in EVERY subset and a stable sign:
  fees/serv      r from -0.48 to -0.70


## 3. How much of this is compositional?

The complaint shares are cut from one total: $s_c = x_c / n$ with $\sum_c s_c \leq 1$ (82.3%
of complaints on average, and exactly 1.0 for many lenders). Two consequences follow, and
they point in **opposite** directions:

- A lender with a high rate in one dimension has mechanically less rate available for the
  others. Negative correlation is therefore partly arithmetic.
- The same coupling means a lender who is genuinely bad at everything shows up as high on
  *several* dimensions at once. Negative correlation therefore **masks** redundancy rather
  than creating it.

So the honest test is to compare a **closed** representation (shares, and the rate-based Method
C scores) against an **unclosed** one (raw counts $x_c$, which have no sum-to-one constraint).
If the counts are positively correlated while the shares are negatively correlated, the
negative sign is compositional and the positive overlap is real.

### 3a. Counts vs shares vs posterior means vs scores

In [5]:
def pair_series(mat, m=None):
    d = mat if m is None else mat[m]
    cm = d.corr().to_numpy()
    return pd.Series([cm[a, b] for a, b in zip(*IU)], index=PAIR)

cmp_all = pd.DataFrame({
    "counts": pair_series(X),
    "shares": pair_series(W),
    "posterior": pair_series(P),
    "C_scores": pair_series(S)})
print("Pearson, all 482 lenders, four representations of the same five dimensions")
print(cmp_all.round(3).to_string())
print(f"\nmean off-diagonal r   counts {cmp_all['counts'].mean():+.3f}   "
      f"shares {cmp_all['shares'].mean():+.3f}   "
      f"posterior {cmp_all['posterior'].mean():+.3f}   "
      f"C scores {cmp_all['C_scores'].mean():+.3f}")
print(f"all 10 count correlations positive: {bool((cmp_all['counts'] > 0).all())}")
print(f"all 10 share correlations negative: {bool((cmp_all['shares'] < 0).all())}")
print("\nRaw counts are NOT closed, so their correlations are the cleanest available")
print("measure of genuine co-occurrence. They are positive in every single pair, ranging")
print(f"{cmp_all['counts'].min():+.2f} to {cmp_all['counts'].max():+.2f}. The share and score")
print("representations are closed, and both go negative. The sign flip is the closure")
print("signature: the same lenders generate both complaint types, and expressing that as a")
print("share of a fixed total forces the negative sign.")

print("\n=== a centred log-ratio was considered and rejected as a diagnostic ===")
_lg = np.log(X.to_numpy() + 0.5)
_clr = _lg - _lg.mean(axis=1, keepdims=True)
_cl = pd.DataFrame(_clr, columns=SLUG)
_clr_mean = pair_series(_cl).mean()
print(f"mean off-diagonal clr correlation = {_clr_mean:+.4f}")
print("For k components summing to zero the off-diagonal correlations are forced to average")
print(f"-1/(k-1) = {-1/(len(SLUG)-1):+.4f} whatever the data say, so a clr correlation cannot")
print("answer 'is there positive association'. The unclosed counts can, so they are used.")

print("\n=== same comparison restricted to the informative subset ===")
m20 = n >= 20
cmp20 = pd.DataFrame({"counts": pair_series(X, m20),
                      "shares": pair_series(W, m20),
                      "C_scores": pair_series(S, m20)})
print(cmp20.round(3).to_string())

Pearson, all 482 lenders, four representations of the same five dimensions
           counts  shares  posterior  C_scores
with/fees   0.275  -0.151     -0.217    -0.124
with/unau   0.269  -0.153     -0.251    -0.222
with/cred   0.124  -0.123     -0.249    -0.287
with/serv   0.443  -0.151     -0.086    -0.003
fees/unau   0.578  -0.218     -0.361    -0.339
fees/cred   0.359  -0.170     -0.311    -0.295
fees/serv   0.590  -0.281     -0.488    -0.481
unau/cred   0.658  -0.074      0.194     0.305
unau/serv   0.817  -0.211      0.030     0.142
cred/serv   0.631  -0.164      0.021     0.130

mean off-diagonal r   counts +0.474   shares -0.170   posterior -0.172   C scores -0.117
all 10 count correlations positive: True
all 10 share correlations negative: True

Raw counts are NOT closed, so their correlations are the cleanest available
measure of genuine co-occurrence. They are positive in every single pair, ranging
+0.12 to +0.82. The share and score
representations are closed, and both go n

### 3b. The score transform hides part of the dependence

`Score = 100 * F(peer | x + a, n - x + b)` is a monotone but **nonlinear** squashing of the
latent posterior z-score of the peer rate. The CDF is steep in the middle and flat in both
tails, so two lenders at very different z-values can receive similar scores, and the
correlation of the scores is smaller than the correlation of the z-values they came from.

This matters for the audit: **redundancy measured on the scores is a lower bound.** Where the
two disagree, the z-correlation is the better estimate of how much shared signal exists.

In [6]:
att = pd.DataFrame({"C_r": pair_series(S), "z_r": pair_series(Z)})
att["ratio"] = att.C_r / att.z_r
print("all 482 lenders: dependence in score space vs in latent posterior-z space")
print(att.round(3).to_string())
print(f"\nmedian |r| in z space {att['z_r'].abs().median():.3f} vs "
      f"{att['C_r'].abs().median():.3f} in score space")
for pair in ["unau/cred", "unau/serv", "cred/serv", "fees/serv"]:
    row = att.loc[pair]
    print(f"  {pair:<10} z r={row['z_r']:+.3f} -> C r={row['C_r']:+.3f} "
          f"(score retains {100*abs(row['ratio']):.0f}% of the latent association)")
print("\nThe C-score correlation is reported in section 2 because that is the object the")
print("label will display. The z-correlation is reported here so the redundancy question is")
print("not answered on an artefact of the display transform.")

all 482 lenders: dependence in score space vs in latent posterior-z space
             C_r    z_r  ratio
with/fees -0.124 -0.306  0.404
with/unau -0.222 -0.540  0.412
with/cred -0.287 -0.453  0.633
with/serv -0.003 -0.174  0.018
fees/unau -0.339 -0.244  1.393
fees/cred -0.295 -0.429  0.688
fees/serv -0.481 -0.561  0.858
unau/cred  0.305  0.586  0.520
unau/serv  0.142  0.434  0.327
cred/serv  0.130  0.485  0.269

median |r| in z space 0.443 vs 0.255 in score space
  unau/cred  z r=+0.586 -> C r=+0.305 (score retains 52% of the latent association)
  unau/serv  z r=+0.434 -> C r=+0.142 (score retains 33% of the latent association)
  cred/serv  z r=+0.485 -> C r=+0.130 (score retains 27% of the latent association)
  fees/serv  z r=-0.561 -> C r=-0.481 (score retains 86% of the latent association)

The C-score correlation is reported in section 2 because that is the object the
label will display. The z-correlation is reported here so the redundancy question is
not answered on an artefact of

## 4. Redundancy: which pairs carry the same signal?

Three tests are applied, in increasing strictness.

1. **Unclosed co-occurrence.** Do raw counts of the two categories rise together, and does
   that survive removing `log1p(n_complaints)`? Volume is the obvious confounder: big lenders
   generate more of everything, so a raw count correlation can be pure size.
2. **Score-space persistence.** Is the pair still large in Method C score space, in the same
   sign, in every volume subset?
3. **Inspection.** For each surviving pair, look at actual lenders side by side and decide
   whether both categories are substantively present, or whether one is dragging the other.

Nothing is merged. Flagging is for the label designer, not for this notebook.

In [7]:
ln_n = np.log1p(n)

def partial_count_corr(a, b, mask=None):
    # Correlation of the two count columns after removing log1p(n_complaints).
    d1, d2 = X[SLUG[a]].to_numpy(float), X[SLUG[b]].to_numpy(float)
    v = ln_n
    if mask is not None:
        d1, d2, v = d1[mask], d2[mask], v[mask]
    A_ = np.column_stack([np.ones(len(d1)), v])
    r1 = d1 - A_ @ np.linalg.lstsq(A_, d1, rcond=None)[0]
    r2 = d2 - A_ @ np.linalg.lstsq(A_, d2, rcond=None)[0]
    return float(np.corrcoef(r1, r2)[0, 1])

flag = []
for a, b in zip(*IU):
    pr = f"{SHORT[SLUG[a]]}/{SHORT[SLUG[b]]}"
    cnt = float(np.corrcoef(X[SLUG[a]], X[SLUG[b]])[0, 1])
    par = partial_count_corr(a, b)
    par20 = partial_count_corr(a, b, n >= 20)
    cs = [S[m].corr().iloc[a, b] for _, m in SUBSETS]
    flag.append({"pair": pr, "count_r": cnt, "count_r_partial_n": par,
                 "count_r_partial_n_ge20": par20,
                 "C_min_abs": min(abs(v) for v in cs),
                 "C_sign_stable": len({np.sign(v) for v in cs if abs(v) > 0.05}) == 1,
                 "z_r": float(pair_series(Z)[pr])})
flag = pd.DataFrame(flag)
flag["volume_robust_overlap"] = (flag["count_r_partial_n"] > 0.3) & (flag["count_r_partial_n_ge20"] > 0.3)
print("volume-robust overlap = count correlation stays above 0.30 after removing log1p(n),")
print("both on all lenders and on the >=20 subset")
print(flag.round(3).to_string(index=False))
print("\nPairs flagged as carrying a shared, volume-independent signal:")
for _, r in flag[flag.volume_robust_overlap].iterrows():
    print(f"  {r['pair']:<12} count r={r['count_r']:+.2f} -> partial {r['count_r_partial_n']:+.2f} "
          f"(>=20: {r['count_r_partial_n_ge20']:+.2f}) | C r in [{r['C_min_abs']:.2f} abs, "
          f"stable={r['C_sign_stable']}] | z r={r['z_r']:+.2f}")

volume-robust overlap = count correlation stays above 0.30 after removing log1p(n),
both on all lenders and on the >=20 subset
     pair  count_r  count_r_partial_n  count_r_partial_n_ge20  C_min_abs  C_sign_stable    z_r  volume_robust_overlap
with/fees    0.275             -0.015                  -0.502      0.124           True -0.306                  False
with/unau    0.269              0.029                  -0.205      0.043           True -0.540                  False
with/cred    0.124             -0.113                  -0.269      0.287           True -0.453                  False
with/serv    0.443              0.231                  -0.018      0.003           True -0.174                  False
fees/unau    0.578              0.291                  -0.076      0.339           True -0.244                  False
fees/cred    0.359              0.013                  -0.295      0.212           True -0.429                  False
fees/serv    0.590              0.201          

### 4a. Eigenvalue spectrum of the score correlation matrix

A compact redundancy measure that needs no pair-by-pair judgement. If the five dimensions were
independent the five eigenvalues would all be 1.0, i.e. 20% of the variance each. A large
first eigenvalue means a common factor that all five share; eigenvalues below 1 mean residual
dimensions that are comparatively distinct.

This is reported as a property of the **correlation matrix only**. No per-lender projection
onto any component is computed, and no composite is formed.

In [8]:
print("eigenvalue spectrum of the Method C score correlation matrix")
spec = []
for lab, m in SUBSETS:
    ev = np.linalg.eigvalsh(S[m].corr().to_numpy())[::-1]
    spec.append({"subset": lab, "k": int(m.sum()),
                 "PC1_%": 100*ev[0]/5, "PC2_%": 100*ev[1]/5,
                 "PC1+PC2_%": 100*ev[:2].sum()/5,
                 "min_eig": ev[-1],
                 "eig": np.round(ev, 2)})
spec = pd.DataFrame(spec)
print(spec.drop(columns=["eig"]).round(2).to_string(index=False))
for _, r in spec.iterrows():
    print(f"  {r['subset']:<14} eigenvalues {r['eig']}")
print(f"\nindependence would give exactly 20.0% on every component.")
print(f"PC1 carries {spec['PC1_%'].min():.0f}-{spec['PC1_%'].max():.0f}% of the variance, and "
      f"PC1+PC2 {spec['PC1+PC2_%'].min():.0f}-{spec['PC1+PC2_%'].max():.0f}%.")
print("So there IS a common factor, and it strengthens as the sample tightens to lenders")
print("whose scores actually discriminate. But it is a factor of roughly 2 dimensions, not a")
print("single dominant one, which is why the five are not simply five copies of one number.")

print("\n=== same spectrum for the unclosed counts (the substantive structure) ===")
for lab, m in SUBSETS:
    ev = np.linalg.eigvalsh(X[m].corr().to_numpy())[::-1]
    print(f"  {lab:<14} k={int(m.sum()):3d} PC1={100*ev[0]/5:5.1f}%  PC1+PC2={100*ev[:2].sum()/5:5.1f}%")

eigenvalue spectrum of the Method C score correlation matrix
         subset   k  PC1_%  PC2_%  PC1+PC2_%  min_eig
        all 482 482  37.94  25.58      63.52     0.39
>=10 complaints 100  46.03  27.48      73.51     0.37
>=20 complaints  70  47.58  28.21      75.79     0.34
>=50 complaints  32  47.63  29.06      76.69     0.26
  all 482        eigenvalues [1.9  1.28 0.76 0.68 0.39]
  >=10 complaints eigenvalues [2.3  1.37 0.49 0.47 0.37]
  >=20 complaints eigenvalues [2.38 1.41 0.46 0.41 0.34]
  >=50 complaints eigenvalues [2.38 1.45 0.54 0.37 0.26]

independence would give exactly 20.0% on every component.
PC1 carries 38-48% of the variance, and PC1+PC2 64-77%.
So there IS a common factor, and it strengthens as the sample tightens to lenders
whose scores actually discriminate. But it is a factor of roughly 2 dimensions, not a
single dominant one, which is why the five are not simply five copies of one number.

=== same spectrum for the unclosed counts (the substantive structure) ===

### 4b. Inspection: overlap or distinct behaviour?

For each flagged pair, representative lenders are shown with each dimension side by side. The
question is whether a lender who is genuinely weak on both is weak on both *in substance*, or
whether one category is simply high because the other is low.

Inspection rows are individual lenders across dimensions. They are deliberately **not** summed,
ranked or combined.

In [9]:
def show(df, cols, title):
    print(f"\n--- {title} ---")
    print(df[cols].to_string(index=False))

# Flagged pair 1: unauthorized / credit reporting, the only pair positive in every subset.
both_worst = ((S["unauthorized"] <= S["unauthorized"].quantile(.25)) &
              (S["credit_rep"] <= S["credit_rep"].quantile(.25))).to_numpy()
print(f"lenders in the bottom quartile on BOTH unauthorized and credit reporting: {int(both_worst.sum())}")
rows = []
for s in SLUG:
    rows.append({"dimension": DIMS[s],
                 "median count, bottom-quartile-on-both": float(X[s][both_worst].median()),
                 "median count, all lenders": float(X[s].median())})
show(pd.DataFrame(rows), ["dimension", "median count, bottom-quartile-on-both",
                          "median count, all lenders"],
     "are BOTH categories actually present in the co-affected lenders?")

sel = np.where(n >= 20)[0]
order = sel[np.argsort(S["unauthorized"].to_numpy()[sel] + S["credit_rep"].to_numpy()[sel])]
insp = pd.DataFrame({
    "company": t["canonical_company"].to_numpy()[order[:6]],
    "n_complaints": n[order[:6]].astype(int),
    "x_unauthorized": X["unauthorized"].to_numpy()[order[:6]].astype(int),
    "x_credit_rep": X["credit_rep"].to_numpy()[order[:6]].astype(int),
    "C_unauthorized": S["unauthorized"].to_numpy()[order[:6]].round(1),
    "C_credit_rep": S["credit_rep"].to_numpy()[order[:6]].round(1)})
show(insp, list(insp.columns),
     "six lenders worst on both, n>=20: both counts are substantial in the same lender")

# Flagged pair 2: fees / servicing, the strongest negative pair.
print(f"\n--- fees / servicing: is the negative sign a trade-off or arithmetic? ---")
for lab, msk in [("bottom quartile of C_fees", S["fees"] <= S["fees"].quantile(.25)),
                 ("top quartile of C_servicing", S["servicing"] >= S["servicing"].quantile(.75))]:
    print(f"  {lab:<26} n={int(msk.sum()):3d}  median x_fees={float(X['fees'][msk].median()):5.1f}  "
          f"median x_servicing={float(X['servicing'][msk].median()):5.1f}  "
          f"median n={float(np.median(n[msk])):5.1f}")
print("  Lenders strong on servicing are not strong on fees, but they are not weak on fees")
print("  either; the pair is a rate trade-off inside a shared denominator rather than a")
print("  claim that the two complaint types are substitutes in substance.")

lenders in the bottom quartile on BOTH unauthorized and credit reporting: 67

--- are BOTH categories actually present in the co-affected lenders? ---
                       dimension  median count, bottom-quartile-on-both  median count, all lenders
  withdrawal_and_payment_control                                    0.0                        0.0
                  fees_and_costs                                    0.0                        0.0
unauthorized_or_unrequested_loan                                    1.0                        0.0
                credit_reporting                                    1.0                        0.0
  servicing_and_payment_handling                                    1.0                        1.0

--- six lenders worst on both, n>=20: both counts are substantial in the same lender ---
                                    company  n_complaints  x_unauthorized  x_credit_rep  C_unauthorized  C_credit_rep
     TRANSUNION INTERMEDIATE HOLDINGS, INC.    

## 5. Does complaint volume drive the scores?

Each dimension score is correlated with `n_complaints` and `log1p(n_complaints)`. The question
is whether volume systematically flatters or penalises a lender.

A weak correlation is expected and is not by itself reassuring, because a weak correlation
could also mean two offsetting effects. The mechanism is therefore identified explicitly: the
score compares a lender's rate to a **fixed** peer rate, so the sign of the volume correlation
is set by where that peer rate sits relative to the rate that high-volume lenders typically
have. If that is the whole story, the sign is predictable per dimension and small.

In [10]:
vol = []
for s in SLUG:
    r = peer_rate(s)
    rate = t[f"share_{s}"].to_numpy(float)
    med_50 = float(np.median(rate[n >= 50]))
    vol.append({"dimension": DIMS[s],
                "peer_rate": r,
                "median rate, 50+ band": med_50,
                "peer vs 50+ typical": "peer ABOVE" if r > med_50 else "peer BELOW",
                "r_vs_n": float(np.corrcoef(S[s], n)[0, 1]),
                "r_vs_log1p_n": float(np.corrcoef(S[s], ln_n)[0, 1]),
                "spearman_vs_n": float(spearmanr(S[s], n).statistic)})
vol = pd.DataFrame(vol)
print(vol.round(3).to_string(index=False))
print(f"\nlargest |r| against volume, Pearson: {vol['r_vs_n'].abs().max():.3f}")
print(f"largest |r| against volume, Spearman: {vol['spearman_vs_n'].abs().max():.3f}")
print(f"signs of r_vs_n: {sorted(np.sign(vol['r_vs_n']).astype(int))}")
print("\nThe sign of the volume correlation matches the 'peer vs 50+ typical' column in every")
print("dimension: where the peer rate sits above what large lenders typically post, large")
print("dimension, with no exceptions. Where the peer rate sits ABOVE what large lenders")
print("typically post, a large lender's rate falls below the peer rate and it scores HIGH,")
print("giving a positive correlation with volume; where the peer rate sits BELOW, the sign")
print("flips. The mechanism is reference placement, not a volume penalty, and every")
print("magnitude is under 0.20.")

print("\n=== band means, to confirm no volume group is systematically flattered ===")
BANDS = [("<10", n < 10), ("10-19", (n >= 10) & (n < 20)),
         ("20-49", (n >= 20) & (n < 50)), ("50+", n >= 50)]
# rows = dimensions, columns = volume bands
band = pd.DataFrame({lab: S.loc[m].mean() for lab, m in BANDS})
band.index = [DIMS[s] for s in band.index]
print(band.round(2).to_string())
print(f"\nrange of band means: {band.to_numpy().min():.1f} to {band.to_numpy().max():.1f}")

                       dimension  peer_rate  median rate, 50+ band peer vs 50+ typical  r_vs_n  r_vs_log1p_n  spearman_vs_n
  withdrawal_and_payment_control      0.068                  0.081          peer BELOW  -0.108        -0.095          0.144
                  fees_and_costs      0.207                  0.274          peer BELOW  -0.145        -0.150         -0.003
unauthorized_or_unrequested_loan      0.095                  0.082          peer ABOVE   0.130         0.154          0.233
                credit_reporting      0.041                  0.018          peer ABOVE   0.149         0.110          0.329
  servicing_and_payment_handling      0.241                  0.168          peer ABOVE   0.189         0.137          0.063

largest |r| against volume, Pearson: 0.189
largest |r| against volume, Spearman: 0.329
signs of r_vs_n: [-1, -1, 1, 1, 1]

The sign of the volume correlation matches the 'peer vs 50+ typical' column in every
dimension: where the peer rate sits above what 

## 6. What would combining the five actually cost?

Not a proposal to combine - a measurement of how much an unweighted combination would keep,
purely as a dependence statistic computed from the correlation matrix. No composite is formed
and no lender-level aggregate is produced.

If the five were independent, the sum of their correlations would be 0 and an equal-weight
average would retain all of the variance. Here the off-diagonal correlations sum to strongly
negative, so the dimensions partially cancel: the average of the five would be a much noisier
estimate of anything than the five shown separately.

This is the direct statistical answer to whether the five can later be combined, and it is a
property of the correlation structure, not a weighting choice.

In [11]:
print("information retained by an equal-weight average of the five scores")
print("(a dependence statistic from the correlation matrix; no composite is computed)\n")
ret = []
for lab, m in SUBSETS:
    Cm = S[m].corr().to_numpy()
    k = len(SLUG)
    off = Cm[np.triu_indices(k, 1)].sum()
    var_actual = (k + 2 * off) / k ** 2
    ret.append({"subset": lab, "k": int(m.sum()),
                "sum_offdiag_r": off,
                "mean_offdiag_r": off / (k * (k - 1) / 2),
                "variance_retained": var_actual,
                "share_lost": 1 - var_actual})
ret = pd.DataFrame(ret)
print(ret.round(3).to_string(index=False))
print("\n`variance_retained` = 1.0 would mean the five carry five independent pieces of")
print("information. Values near 0.1 mean an unweighted average would throw away roughly 90%")
print("of the signal, because the negative pairs cancel the positive ones.")
print("\nThe negative cancellations are strongest on all 482 lenders and weaken as the")
print("subset tightens, because the closure effect is strongest where shares are smallest.")

information retained by an equal-weight average of the five scores
(a dependence statistic from the correlation matrix; no composite is computed)

         subset   k  sum_offdiag_r  mean_offdiag_r  variance_retained  share_lost
        all 482 482         -1.174          -0.117              0.106       0.894
>=10 complaints 100         -0.838          -0.084              0.133       0.867
>=20 complaints  70         -0.771          -0.077              0.138       0.862
>=50 complaints  32         -0.483          -0.048              0.161       0.839

`variance_retained` = 1.0 would mean the five carry five independent pieces of
information. Values near 0.1 mean an unweighted average would throw away roughly 90%
of the signal, because the negative pairs cancel the positive ones.

The negative cancellations are strongest on all 482 lenders and weaken as the
subset tightens, because the closure effect is strongest where shares are smallest.


## 7. Assertions

In [12]:
res = []

def claim(label, got, expected, tol=5e-3):
    ok = abs(float(got) - float(expected)) <= tol
    res.append((label, float(got), float(expected), float(tol), "PASS" if ok else "FAIL"))

def claim_bool(label, ok):
    res.append((label, float(bool(ok)), 1.0, 0.0, "PASS" if ok else "FAIL"))

# --- Method C reproduced exactly: peer rates from the fitted Beta medians
for s, r in [("withdrawal", 0.0684), ("fees", 0.2075), ("unauthorized", 0.0954),
             ("credit_rep", 0.0406), ("servicing", 0.2414)]:
    claim(f"peer rate :: {s}", peer_rate(s), r, 5e-4)

# --- and the resulting scores match values published in cfpb_dimension_scoring.ipynb
_i = int(t.index[t["canonical_company"] == "Universal Account Servicing, LLC"][0])
for s, v in [("withdrawal", 9.711), ("fees", 58.754), ("unauthorized", 53.680),
             ("credit_rep", 53.032), ("servicing", 56.737)]:
    claim(f"Method C reproduced :: UAS n=1 :: {s}", S[s].iloc[_i], v, 5e-3)
_i = int(t.index[t["canonical_company"] == "Albert Corporation"][0])
for s, v in [("withdrawal", 0.000), ("fees", 56.766), ("unauthorized", 99.834),
             ("credit_rep", 99.210), ("servicing", 94.740)]:
    claim(f"Method C reproduced :: Albert n=85 :: {s}", S[s].iloc[_i], v, 5e-3)

# --- inputs and shapes
claim("lenders", L, 482, 0)
claim("total complaints", int(n.sum()), 6024, 0)
claim("score matrix shape", S.shape[0] * S.shape[1], 482 * 5, 0)
claim_bool("all five score columns finite and within 0-100",
           bool(np.isfinite(S.to_numpy()).all() and S.to_numpy().min() >= 0
                and S.to_numpy().max() <= 100))
claim_bool("n_complaints retained alongside the scores", n.shape[0] == 482)

# --- the closure diagnosis
claim("mean off-diagonal r, raw COUNTS (unclosed)", pair_series(X).mean(), 0.474, 0.002)
claim("mean off-diagonal r, raw SHARES (closed)", pair_series(W).mean(), -0.170, 0.002)
claim_bool("all 10 count correlations are positive", bool((pair_series(X) > 0).all()))
claim_bool("all 10 share correlations are negative", bool((pair_series(W) < 0).all()))
claim("mean clr off-diagonal r is forced to -1/(k-1)",
      pair_series(_cl).mean(), -0.25, 0.005)

# --- the flagged pair
claim("unau/cred Pearson, all lenders", S.corr().loc["unauthorized", "credit_rep"], 0.305, 0.002)
claim("unau/cred Pearson, >=50", S[n >= 50].corr().loc["unauthorized", "credit_rep"], 0.555, 0.003)
claim("unau/cred count r survives partialling log1p(n)",
      float(flag.loc[flag.pair == "unau/cred", "count_r_partial_n"].iloc[0]), 0.512, 0.003)
claim("fees/serv Pearson, all lenders", S.corr().loc["fees", "servicing"], -0.481, 0.002)
claim("fees/serv Pearson, >=50", S[n >= 50].corr().loc["fees", "servicing"], -0.698, 0.003)
claim("with/unau is NOT stable (Pearson >=20 vs >=50)",
      abs(float(stab.loc[stab.pair == "with/unau", "r_ge20"].iloc[0]
                - stab.loc[stab.pair == "with/unau", "r_ge50"].iloc[0])), 0.21, 0.01)

# --- score transform attenuates the latent dependence
claim("latent z r, unau/cred", pair_series(Z)["unau/cred"], 0.586, 0.003)
claim_bool("C-score r understates latent z r for the 3 flagged pairs",
           all(abs(pair_series(S)[p]) < abs(pair_series(Z)[p])
               for p in ["unau/cred", "unau/serv", "cred/serv"]))

# --- redundancy sizing
claim("PC1 variance share, all lenders", float(spec.loc[spec.subset == "all 482", "PC1_%"].iloc[0]),
      37.9, 0.1)
claim("PC1 variance share, >=20", float(spec.loc[spec.subset == ">=20 complaints", "PC1_%"].iloc[0]),
      47.6, 0.1)
claim_bool("PC1 exceeds the 20% independence baseline in every subset",
           bool((spec["PC1_%"] > 20).all()))

# --- volume relationship is weak and sign-explained
claim("max |r| of any dimension score vs n_complaints", vol["r_vs_n"].abs().max(), 0.189, 0.002)
claim("max |r| vs log1p(n_complaints)", vol["r_vs_log1p_n"].abs().max(), 0.154, 0.002)
claim_bool("volume-correlation sign matches peer-vs-large-lender placement in all 5 dims",
           all((r > 0) == (row["peer vs 50+ typical"] == "peer ABOVE")
               for r, (_, row) in zip(vol["r_vs_n"], vol.iterrows())))

# --- combination cost
claim("sum off-diagonal r, all lenders", float(ret.loc[ret.subset == "all 482", "sum_offdiag_r"].iloc[0]),
      -1.174, 0.003)
claim("variance retained by an equal-weight average, all lenders",
      float(ret.loc[ret.subset == "all 482", "variance_retained"].iloc[0]), 0.106, 0.003)
claim_bool("an equal-weight average loses more than 80% of the variance in every subset",
           bool((ret["share_lost"] > 0.80).all()))

# --- constraints respected
claim_bool("no composite / overall / weighted score column was created",
           not any(k in {c.lower() for c in S.columns} for k in
                   ("composite", "overall", "total", "mean_score", "weight", "rank", "grade")))
claim_bool("processed CSV untouched", IN.exists())
claim_bool("only the five per-dimension score columns exist",
           list(S.columns) == SLUG)

chk = pd.DataFrame(res, columns=["claim", "got", "expected", "tol", "status"])
chk["got"] = chk["got"].round(6)
print(chk.to_string(index=False))
n_fail = int((chk["status"] == "FAIL").sum())
print(f"\n{(chk['status']=='PASS').sum()}/{len(chk)} claims verified   ({n_fail} FAILED)")
if n_fail:
    print("\nFAILED:")
    print(chk[chk["status"] == "FAIL"].to_string(index=False))

                                                                       claim         got  expected    tol status
                                                     peer rate :: withdrawal    0.068421    0.0684 0.0005   PASS
                                                           peer rate :: fees    0.207497    0.2075 0.0005   PASS
                                                   peer rate :: unauthorized    0.095443    0.0954 0.0005   PASS
                                                     peer rate :: credit_rep    0.040640    0.0406 0.0005   PASS
                                                      peer rate :: servicing    0.241351    0.2414 0.0005   PASS
                                Method C reproduced :: UAS n=1 :: withdrawal    9.711233    9.7110 0.0050   PASS
                                      Method C reproduced :: UAS n=1 :: fees   58.753700   58.7540 0.0050   PASS
                              Method C reproduced :: UAS n=1 :: unauthorized   53.679735   53.68

## 8. Answers

### 1. Are the five dimensions sufficiently distinct to appear separately on the FinePrint label?

**Yes, with one caveat.** Four of the five behave as substantially distinct complaint
phenomena. `withdrawal_and_payment_control` is close to orthogonal to everything
(min |r| = 0.00 in score space), and `fees_and_costs`, `servicing_and_payment_handling` and
`withdrawal_and_payment_control` are related mainly through the shared denominator rather than
through shared substance - their count correlations collapse from +0.28 to +0.59 down to
between -0.02 and +0.23 once `log1p(n_complaints)` is removed.

The caveat is that the five are **not** five independent facts. The first principal component
of the score correlation matrix carries 38-48% of the variance against a 20% independence
baseline, and the unclosed counts are positively correlated in all ten pairs (mean +0.47,
up to +0.82). Large lenders are worse on several dimensions at once. That is a real common
factor - general complaint propensity - and displaying five separate scores will let a
consumer add it up five times.

### 2. Which dimensions risk double-counting the same underlying signal?

**`unauthorized_or_unrequested_loan` and `credit_reporting`** are the one pair that should be
treated as a single block. They are the only pair positively correlated in *every* volume
subset, and the correlation *strengthens* as volume rises: +0.30 on all lenders, +0.50 at
10+, +0.54 at 20+, +0.56 at 50+ (Spearman +0.35 rising to +0.66). Crucially it is not a
volume artefact - the raw count correlation is +0.66 and stays at +0.51 after removing
`log1p(n_complaints)` (+0.46 on the 20+ subset). Inspection confirms substance rather than
arithmetic: lenders weak on both carry substantial counts in both, not one high and one zero.

Two further pairs - `unauthorized`/`servicing` and `credit_rep`/`servicing` - have the
strongest volume-robust count overlap of all (+0.70 and +0.44 partial), but in score space they
stay at only +0.13 to +0.45. These are the same underlying factor, diluted differently by the
score transform rather than cleanly distinct.

No pair should be merged on the strength of this analysis. The flag is for the label designer.

### 3. Does complaint volume materially influence the dimension scores?

**No.** Every correlation between a dimension score and complaint volume is under 0.20
(Pearson -0.15 to +0.19, Spearman -0.00 to +0.33), and the signs differ by dimension rather
than pointing one way. The sign is fully explained by reference placement, with no exceptions:
where the peer rate sits above what large lenders typically post, a large lender's rate lands
below the peer rate and it scores high, and the converse holds where the peer rate sits below.
That is a property of where the peer reference was drawn, not a volume penalty built into the
score. Band means stay within 35-63 across the four
volume bands, so no group is systematically flattered or punished.

One caveat to carry forward: the peer rate is low in `credit_reporting` (0.041) relative to
what 50+ lenders typically post (0.018), which is why that dimension mildly rewards volume.
The magnitude is small, but it is a real directional effect and the peer rates are a modelling
choice worth revisiting if volume bias becomes a concern.

### 4. Is there statistical justification for combining all five later?

**No, not as an equal-weight combination of these five scores.** Section 6 measures the cost
directly: the off-diagonal correlations sum to -1.17 on all lenders, so an unweighted average
would retain about 11% of the variance it would have if the dimensions were independent -
roughly 86-89% of the signal is lost to cancellation, and the loss persists (over 80%) in
every subset. The reason is that several pairs are strongly *negatively* related, so averaging
them measures complaint-mix evenness rather than complaint severity.

The dependence structure is also close to two-dimensional rather than one-dimensional
(PC1+PC2 = 64-77% of variance), so a single scalar would be a lossy summary even before
weights are considered.

The defensible reading is that these are **five informational dimensions, not five inputs to
one number**. If a composite is ever required, the analysis says the design question is which
dimensions belong in the same block - starting with `unauthorized` and `credit_reporting` -
not what to weight them. No weights are proposed here and none should be read into the
pairings above.

### What this notebook deliberately does not produce

No overall score, no weights, no lender ranking or grade, and no per-lender composite of the
five dimensions. The only lender-level rows are side-by-side dimension-by-dimension
inspections of named lenders. No file was written; `payday_shrunk_features.csv` is unchanged.